# Embedding Surgery — Part 1 (Finance domain)

**Base model:** `sentence-transformers/all-MiniLM-L6-v2` (general-domain sentence embedding model, 30,522-token WordPiece vocab).

**Goal:** the base tokenizer has never seen core finance terms like *securitization* or *counterparty* as whole tokens — it shatters them into subword fragments. We surgically expand the vocabulary so each domain term becomes a single token with its own embedding, without disturbing anything the model already knows.

**How the tokens were chosen:** we trained a 10k WordPiece vocabulary on a finance corpus (`Finance_vocabulary.txt`), diffed it against the base vocab with `compare_vocabs.py` (→ `finance_new_vocab.txt`, 4,398 missing tokens), and hand-picked 15 distinctly financial whole words from that diff (`domain_tokens.txt`).

**The two key decisions (and their justification):**
1. **Old tokens keep their pretrained embeddings, untouched.** Those vectors encode everything the model learned in pretraining; there is no reason to modify them, and `resize_token_embeddings` preserves them by construction. We verify this bit-for-bit below.
2. **New tokens are initialized as the mean of their old subword embeddings.** Before surgery, `securitization` was represented by the pieces `sec ##uri ##ti ##zation`; averaging those vectors gives the new token a starting point that already sits in the right neighborhood of the embedding space, instead of random noise that would embed domain sentences arbitrarily.


In [1]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID)
model.eval()

print(f"Model: {MODEL_ID}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Embedding matrix shape: {tuple(model.get_input_embeddings().weight.shape)}")

C:\Users\adame\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\adame\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\adame\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10284.32it/s]

Model: sentence-transformers/all-MiniLM-L6-v2
Vocab size: 30522
Embedding matrix shape: (30522, 384)


## 1. The problem — before surgery

Each domain term gets shattered into generic subword fragments:

In [2]:
with open("domain_tokens.txt", encoding="utf-8") as f:
    domain_tokens = [line.strip() for line in f if line.strip()]

print(f"{len(domain_tokens)} domain tokens to add\n")

# Record how each term tokenizes BEFORE surgery — we also need these
# subword IDs later to build the mean-initialization.
pre_surgery_pieces = {}
for tok in domain_tokens:
    pieces = tokenizer.tokenize(tok)
    pre_surgery_pieces[tok] = tokenizer.convert_tokens_to_ids(pieces)
    print(f"{tok:>18} -> {pieces}")

15 domain tokens to add

    securitization -> ['sec', '##uri', '##ti', '##zation']
    collateralized -> ['collateral', '##ized']
      counterparty -> ['counterpart', '##y']
      underwriting -> ['under', '##writing']
           hedging -> ['he', '##d', '##ging']
         liquidity -> ['liquid', '##ity']
        volatility -> ['vol', '##ati', '##lity']
      amortization -> ['amor', '##ti', '##zation']
       refinancing -> ['ref', '##ina', '##nc', '##ing']
       receivables -> ['rec', '##ei', '##vable', '##s']
         covenants -> ['covenant', '##s']
           tranche -> ['tran', '##che']
            escrow -> ['es', '##crow']
           annuity -> ['ann', '##uity']
         actuarial -> ['act', '##ua', '##rial']


In [3]:
# Sanity check: none of these words exist as a single token yet
unk_id = tokenizer.unk_token_id
assert all(
    tokenizer.convert_tokens_to_ids(t) == unk_id for t in domain_tokens
), "Some token already exists in the base vocab!"
print("Confirmed: all 15 terms are absent from the base vocabulary.")

Confirmed: all 15 terms are absent from the base vocabulary.


## 2. Snapshot the original weights

To prove later that surgery touched nothing it shouldn't have, we keep a full copy of the original embedding matrix.

In [4]:
old_vocab_size = len(tokenizer)
old_embeddings = model.get_input_embeddings().weight.detach().clone()
print(f"Snapshot taken: {tuple(old_embeddings.shape)}")

Snapshot taken: (30522, 384)


## 3. The surgery

Two built-in HuggingFace calls, plus the mean-initialization of the new rows:

In [5]:
# (a) Expand the tokenizer vocabulary
num_added = tokenizer.add_tokens(domain_tokens)
print(f"Added {num_added} tokens -> new vocab size: {len(tokenizer)}")

# (b) Grow the embedding matrix to match (old rows are preserved,
#     new rows get a default random init that we overwrite next)
model.resize_token_embeddings(len(tokenizer))

# (c) Initialize each new row as the mean of the embeddings of the
#     subword pieces the term used to be split into
embedding_matrix = model.get_input_embeddings().weight
with torch.no_grad():
    for tok in domain_tokens:
        new_id = tokenizer.convert_tokens_to_ids(tok)
        piece_ids = pre_surgery_pieces[tok]
        embedding_matrix[new_id] = old_embeddings[piece_ids].mean(dim=0)

print(f"New embedding matrix shape: {tuple(embedding_matrix.shape)}")

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Added 15 tokens -> new vocab size: 30537


New embedding matrix shape: (30537, 384)


## 4. Verification

A good check shows **both** that what we changed is correct **and** that what we didn't change is untouched.

In [6]:
emb = model.get_input_embeddings().weight

# (1) UNCHANGED: all 30,522 original rows are bit-identical to the snapshot
assert torch.equal(emb[:old_vocab_size], old_embeddings), "Old embeddings were modified!"
print(f"[OK] All {old_vocab_size} pretrained embeddings are bit-for-bit unchanged.")

# (2) CHANGED, correctly: every domain term is now ONE token...
for tok in domain_tokens:
    pieces = tokenizer.tokenize(tok)
    assert pieces == [tok], f"{tok} still splits: {pieces}"
print(f"[OK] All {len(domain_tokens)} domain terms now tokenize as a single token.")

# ...and its embedding equals the mean of its old subword embeddings
for tok in domain_tokens:
    new_id = tokenizer.convert_tokens_to_ids(tok)
    expected = old_embeddings[pre_surgery_pieces[tok]].mean(dim=0)
    assert torch.equal(emb[new_id], expected), f"Bad init for {tok}"
print(f"[OK] All {len(domain_tokens)} new embeddings equal the mean of their old subword embeddings.")

[OK] All 30522 pretrained embeddings are bit-for-bit unchanged.
[OK] All 15 domain terms now tokenize as a single token.
[OK] All 15 new embeddings equal the mean of their old subword embeddings.


In [7]:
# Before/after tokenization, side by side
print(f"{'term':>18} | before -> after")
print("-" * 60)
for tok in domain_tokens:
    before = " ".join(tokenizer.convert_ids_to_tokens(pre_surgery_pieces[tok]))
    print(f"{tok:>18} | {before}  ->  {tok}")

              term | before -> after
------------------------------------------------------------
    securitization | sec ##uri ##ti ##zation  ->  securitization
    collateralized | collateral ##ized  ->  collateralized
      counterparty | counterpart ##y  ->  counterparty
      underwriting | under ##writing  ->  underwriting
           hedging | he ##d ##ging  ->  hedging
         liquidity | liquid ##ity  ->  liquidity
        volatility | vol ##ati ##lity  ->  volatility
      amortization | amor ##ti ##zation  ->  amortization
       refinancing | ref ##ina ##nc ##ing  ->  refinancing
       receivables | rec ##ei ##vable ##s  ->  receivables
         covenants | covenant ##s  ->  covenants
           tranche | tran ##che  ->  tranche
            escrow | es ##crow  ->  escrow
           annuity | ann ##uity  ->  annuity
         actuarial | act ##ua ##rial  ->  actuarial


## 5. Save, reload from disk, and embed finance text end-to-end

The final proof: the saved model loads cleanly and produces sensible sentence embeddings on domain text.

In [8]:
SAVE_DIR = "finance-minilm-l6-v2"
tokenizer.save_pretrained(SAVE_DIR)
model.save_pretrained(SAVE_DIR)
print(f"Saved to ./{SAVE_DIR}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 12.91it/s]

Saved to ./finance-minilm-l6-v2


In [9]:
# Reload from disk — completely fresh objects
tok2 = AutoTokenizer.from_pretrained(SAVE_DIR)
model2 = AutoModel.from_pretrained(SAVE_DIR)
model2.eval()

def embed(sentences):
    """Mean-pooled sentence embeddings, as in the all-MiniLM-L6-v2 model card."""
    enc = tok2(sentences, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        out = model2(**enc)
    mask = enc["attention_mask"].unsqueeze(-1).float()
    emb = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
    return torch.nn.functional.normalize(emb, dim=1)

sentences = [
    "The bank improved its liquidity through securitization of receivables.",
    "Counterparty risk and loan covenants are reviewed by the underwriting team.",
    "My cat likes to sleep on the sofa all afternoon.",
]

# The reloaded tokenizer treats domain terms as single tokens
print("Reloaded tokenizer on sentence 1:")
print(tok2.tokenize(sentences[0]), "\n")

E = embed(sentences)
sim = E @ E.T
print(f"similarity(finance, finance) = {sim[0,1]:.3f}")
print(f"similarity(finance, cat)     = {sim[0,2]:.3f}")
assert sim[0, 1] > sim[0, 2], "Domain sentences should be closer to each other than to off-topic text"
print("\n[OK] Reloaded model embeds domain text sensibly end-to-end.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7211.64it/s]

Reloaded tokenizer on sentence 1:
['the', 'bank', 'improved', 'its', 'liquidity', 'through', 'securitization', 'of', 'receivables', '.'] 



similarity(finance, finance) = 0.320
similarity(finance, cat)     = -0.054

[OK] Reloaded model embeds domain text sensibly end-to-end.


## 6. Published on the HuggingFace Hub

The surgically-modified model and expanded tokenizer are published at
**https://huggingface.co/adammoood/finance-minilm-l6-v2**.
Setting `PUSH = True` re-uploads them after any change.

In [ ]:
PUSH = False  # already published; set to True to re-upload
REPO_ID = "adammoood/finance-minilm-l6-v2"

if PUSH:
    tokenizer.push_to_hub(REPO_ID)
    model.push_to_hub(REPO_ID)
print(f"Published at: https://huggingface.co/{REPO_ID}")

Published at: https://huggingface.co/adammoood/finance-minilm-l6-v2


## Summary

| | before | after |
|---|---|---|
| vocab size | 30,522 | 30,537 |
| embedding rows | 30,522 | 30,537 |
| `securitization` | `sec ##uri ##ti ##zation` | `securitization` |

- **Old tokens:** untouched — verified bit-for-bit against a pre-surgery snapshot.
- **New tokens:** initialized as the mean of their former subword embeddings — verified exactly.
- **Loadable:** saved with `save_pretrained`, reloaded from disk, and used to embed finance sentences end-to-end.
